# Supermarket Sales ETL Project
Complete ETL solution using Kaggle dataset, Python, Pandas and SQLite.

In [23]:
!pip install kaggle

## Kaggle API Authentication Setup

This step configures Kaggle API access by securely placing the `kaggle.json` credentials file into the required directory and setting appropriate permissions.

In [31]:
import os
import shutil

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move("/Users/rukhsar/Downloads/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)


## Data Extraction from Kaggle

This step authenticates the Kaggle API and downloads the *Supermarket Sales* dataset directly into the local project directory.  
The dataset is automatically extracted and prepared for further transformation and loading stages.

In [43]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "faresashraf1001/supermarket-sales",
    path="data",
    unzip=True
)

print("Download complete")

Dataset URL: https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales
Download complete


## Load Dataset into Pandas DataFrame

In this step, the extracted Supermarket Sales dataset is loaded into a Pandas DataFrame.  
This allows efficient data inspection, preprocessing, validation, and transformation before loading into the target database.

In [166]:
import pandas as pd
import sqlite3

df = pd.read_csv("data/Super-Market-Analysis.csv")
df.rename(columns={
    'Product line': 'Product_line',
    'Customer type': 'Customer_type',
    'Unit price': 'Unit_price',
    'Tax 5%': 'Tax_5_percent'
}, inplace=True)

df['Date'] = pd.to_datetime(df['Date'])
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df.head()

,Invoice ID,Branch,City,Customer_type,Gender,Product_line,Unit_price,Quantity,Tax_5_percent,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-01-05,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-03-08,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,2019-01-27,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2019-02-08,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3


## Building Product Dimension

To support analytical reporting and schema normalization, a Product dimension table is created.

Key operations:
- Select relevant product attributes
- Remove duplicate records
- Generate surrogate primary key (`product_id`)

In [170]:
Product_Data = (
    df[['Branch', 'City', 'Product_line']]
    .drop_duplicates()
    .reset_index(drop=True)
)

Product_Data['product_id'] = Product_Data.index + 1

## Building Customer Dimension

To support dimensional modeling, customer attributes are separated into a dedicated dimension table.

Key Steps:
- Select customer-related attributes
- Remove duplicate records
- Generaye surrogate primary key (`customer_id`)

In [172]:
Customer_Data = (
    df[['Customer_type', 'Gender', 'Payment']]
    .drop_duplicates()
    .reset_index(drop=True)
)

Customer_Data['customer_id'] = Customer_Data.index + 1


## Building Fact Sales_Data Table (Star Schema Implementation)

Purpose:
- Central table for analytical reporting
- Enables efficient joins with dimensions
- Stores numerical business metrics

Measures Included:
- Quantity
- Unit Price
- Tax
- Total Sales

Dimensions Referenced:
- Product Dimension
- Customer Dimension

In [180]:
# Product_Data.head()
# Customer_Data.head()

fact = df.merge(
    Product_Data,
    on=['Branch', 'City', 'Product_line']
).merge(
    Customer_Data,
    on=['Customer_type', 'Gender', 'Payment']
)

Sales_Data = fact[
    ['product_id','customer_id','Date','Time',
     'Quantity','Unit_price','Tax_5_percent','Sales']
]

Sales_Data.columns = [
    'product_id','customer_id','date','time',
    'quantity','unit_price','tax','sales'
]

## Data Loading – Warehouse Layer Creation

The final ETL stage loads curated datasets into a relational database representing a warehouse layer.

Tables Created:
- **Product_Data** → Product Dimension
- **Customer_Data** → Customer Dimension
- **Sales_Data** → Sales Fact Table

Purpose:
- Enable SQL-based analytics
- Support reporting queries
- Prepare data for aggregation and window functions

In [182]:
conn = sqlite3.connect("sales_dw.db")

Product_Data.to_sql("Product_Data", conn,
                   if_exists="replace", index=False)

Customer_Data.to_sql("Customer_Data", conn,
                    if_exists="replace", index=False)

Sales_Data.to_sql("Sales_Data", conn,
                  if_exists="replace", index=False)

conn.close()

print("Data loaded successfully into SQLite DB")

Data loaded successfully into SQLite DB


# 🏆 City-wise Product & Customer Sales Performance Analysis

## Overview

The **City-wise Product and Customer Sales Performance Analysis** evaluates how different product lines perform across cities and customer segments.

This report combines product, customer, and sales data to identify high-performing combinations that drive revenue.

---

## Business Objective

- Identify top-selling product lines within each city  
- Understand purchasing behavior across customer types  
- Rank products based on sales performance  
- Analyze cumulative revenue contribution  


In [217]:
conn = sqlite3.connect("sales_dw.db")

Sales_Report = '''
SELECT
    p.city AS City,
    p.Product_line AS Product_line,
    c.Customer_type AS Customer_type,

    SUM(f.sales) AS Total_sales,
    SUM(f.quantity) AS Total_quantity,
    AVG(f.sales) AS Avg_sale_value,

    DENSE_RANK() OVER (
        PARTITION BY p.city
        ORDER BY SUM(f.sales) DESC
    ) AS Sales_rank_in_city,

    SUM(f.sales) OVER (
        PARTITION BY p.city
        ORDER BY SUM(f.sales) DESC
    ) AS cumulative_sales

FROM Sales_Data f
JOIN Product_Data p
    ON f.product_id = p.product_id
JOIN Customer_Data c
    ON f.customer_id = c.customer_id

GROUP BY p.city, p.Product_line, c.Customer_type

ORDER BY City, Total_sales DESC
'''

Sales_Report_Data = pd.read_sql(Sales_Report, conn)
Sales_Report_Data

,City,Product_line,Customer_type,Total_sales,Total_quantity,Avg_sale_value,Sales_rank_in_city,cumulative_sales
0,Mandalay,Health and beauty,Member,13235.7015,198,441.190050,1,277.1370
1,Mandalay,Sports and travel,Member,11976.9510,182,374.279719,2,867.5730
2,Mandalay,Food and beverages,Member,10191.5520,168,328.759742,3,1040.3190
3,Mandalay,Electronic accessories,Normal,9626.5890,169,343.806750,4,1982.7675
4,Mandalay,Home and lifestyle,Normal,8958.3375,144,373.264062,5,2246.7375
5,Mandalay,Home and lifestyle,Member,8590.8270,151,330.416423,6,2331.3675
6,Mandalay,Fashion accessories,Member,8404.2840,156,240.122400,7,2392.1835
7,Mandalay,Sports and travel,Normal,8011.2480,140,267.041600,8,2538.5115
8,Mandalay,Fashion accessories,Normal,8009.0325,141,296.630833,9,2829.7185
9,Mandalay,Electronic accessories,Member,7424.8545,147,274.994611,10,2936.8605


# 📊 Top Customers Contribution

## Overview

The **Top Customers Contribution** analysis identifies high-value customers contributing the largest share of total sales revenue.

This helps businesses to:

- Recognize loyal and premium customers  
- Design targeted marketing strategies  
- Improve customer retention programs  
- Focus on revenue-driving customer segments  

---

## Business Objective

- Identify top-performing customers  
- Measure their percentage contribution to overall sales  
- Enable data-driven customer segmentation

In [206]:
Customer_Report = '''SELECT
    c.customer_type AS Customer_type,
    c.gender AS Gender,
    c.payment AS Payment_mode,
    SUM(f.sales) AS Revenue,

    SUM(SUM(f.sales)) OVER(
        ORDER BY SUM(f.sales) DESC
    ) AS Cumulative_revenue,

    ROUND(
        100 * SUM(SUM(f.sales)) OVER(
            ORDER BY SUM(f.sales) DESC
        )
        / SUM(SUM(f.sales)) OVER(),
        2
    ) AS Cumulative_percentage

FROM Sales_data f
JOIN Customer_data c
    ON f.customer_id = c.customer_id

GROUP BY
    c.customer_type,c.gender,c.payment

ORDER BY
    Revenue DESC
LIMIT 10
'''

Top_Customers_Contribution = pd.read_sql(Customer_Report, conn)
Top_Customers_Contribution

,Customer_type,Gender,Payment_mode,Revenue,Cumulative_revenue,Cumulative_percentage
0,Member,Female,Cash,43246.2345,43246.2345,13.39
1,Member,Female,Credit card,41486.3085,84732.5430,26.24
2,Member,Female,Ewallet,40473.5940,125206.1370,38.77
3,Normal,Female,Cash,26101.0050,151307.1420,46.85
4,Normal,Male,Ewallet,24469.9245,175777.0665,54.43
5,Normal,Female,Ewallet,23685.3960,199462.4625,61.76
6,Member,Male,Credit card,22044.0255,221506.4880,68.58
7,Normal,Male,Cash,21778.9215,243285.4095,75.33
8,Member,Male,Ewallet,21364.1925,264649.6020,81.94
9,Member,Male,Cash,21080.4090,285730.0110,88.47


# 📈 Monthly Sales Trend Analysis

## Overview

The **Monthly Sales Trend Analysis** evaluates how sales revenue changes over time across different cities.

This analysis helps in understanding seasonal patterns, business growth, and market performance trends.

---

## Business Objective

- Track month-over-month revenue performance  
- Identify sales growth or decline trends  
- Compare performance across cities  
- Support forecasting and business planning  

In [198]:
Sales_Trend_Report = '''SELECT
    p.city AS City,

    strftime('%Y-%m', f.date) AS Sales_month,

    ROUND(SUM(f.sales),2) AS Monthly_revenue,

    ROUND(IFNULL(
        LAG(SUM(f.sales)) OVER(
            PARTITION BY p.city
            ORDER BY strftime('%Y-%m', f.date)
        ),0
    ),2) AS Previous_month_revenue,

    IFNULL(
        ROUND(SUM(f.sales),2)
        - LAG(ROUND(SUM(f.sales),2)) OVER(
            PARTITION BY p.city
            ORDER BY strftime('%Y-%m', f.date)
        ),0
    ) AS Revenue_growth

FROM Sales_Data f
JOIN Product_Data p
    ON f.product_id = p.product_id

GROUP BY
    p.city,
    Sales_month

ORDER BY
    City,
    Sales_month
'''

Monthly_Sales_Trend_Analysis = pd.read_sql(Sales_Trend_Report, conn)
Monthly_Sales_Trend_Analysis

,City,Sales_month,Monthly_revenue,Previous_month_revenue,Revenue_growth
0,Mandalay,2019-01,37176.06,0.00,0.00
1,Mandalay,2019-02,34424.27,37176.06,-2751.79
2,Mandalay,2019-03,34597.34,34424.27,173.07
3,Naypyitaw,2019-01,40434.68,0.00,0.00
4,Naypyitaw,2019-02,32934.98,40434.68,-7499.70
5,Naypyitaw,2019-03,37199.04,32934.98,4264.06
6,Yangon,2019-01,38681.13,0.00,0.00
7,Yangon,2019-02,29860.12,38681.13,-8821.01
8,Yangon,2019-03,37659.12,29860.12,7799.00
